# Image Classification PyTorch

https://colab.research.google.com/drive/1R1Yj4sGbeWnR2qiGSDvrCHKKVUKBVeF4#scrollTo=cd818528

In [ ]:
import pandas as pd
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from pathlib import Path
from sklearn.model_selection import train_test_split
from torchvision.models import convnext_large
from torch import nn
from glob import glob
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm
from sklearn.metrics import accuracy_score

## Transform Dataset

In [ ]:
class CustomImageDataset(Dataset):

    def __init__(self, dataframe,image_folder, transform=None):
        self.data = dataframe
        self.root = Path(image_folder)
        self.transform = transform
        self.image_paths = self.data['image_name'].values
        self.labels = self.data['class'].values

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # Load image
        img_path = Path(self.image_paths[idx])
        image = Image.open(self.root/img_path).convert('RGB')  # Convert to RGB

        # Load label
        label = self.labels[idx]
        label = torch.tensor(label,dtype=torch.float)  # For classification

        # Apply transformations
        if self.transform:
            image = self.transform(image)

        return image, label.unsqueeze(0)


## Load Dataset

In [ ]:
def download_kaggle(kaggle_command="kaggle competitions download -c liver-fibrosis-severity-prediction"):

  # Get Kaggle Key
  kaggle_username = userdata.get("KAGGLE_USER")
  kaggle_key = userdata.get("KAGGLE_KEY")
  if not kaggle_username or not kaggle_key:
      print("Error: Kaggle_USERNAME or Kaggle_KEY not found in Colab Secrets.")
      return

  # Write the credentials to ~/.kaggle/kaggle.json
  kaggle_dir = os.path.expanduser("~/.kaggle")
  os.makedirs(kaggle_dir, exist_ok=True)

  # Create JSON
  kaggle_json_path = os.path.join(kaggle_dir, "kaggle.json")
  with open(kaggle_json_path, "w") as f:
      f.write(f'{{"username":"{kaggle_username}","key":"{kaggle_key}"}}')
  os.chmod(kaggle_json_path, 0o600)

  try:
      os.system(kaggle_command)
      print("\n--- Download complete! ---")
      os.system("ls -la")
      os.system("unzip -o '*.zip' && rm -f *.zip")
      os.system("ls -la")

  except Exception as e:
      print(f"An error occurred during download: {e}")

In [ ]:
df = pd.read_csv('./data/train.csv')
df.head()

In [ ]:
train_df, eval_df = train_test_split(df, test_size=0.2, random_state=42,stratify=df['class'])
len(train_df), len(eval_df)

In [ ]:
# Create Compose Transform Image
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


# Transform Data
train_ds = CustomImageDataset(dataframe=train_df,
                              image_folder='data/train/train',
                              transform=train_transform)

eval_ds = CustomImageDataset(dataframe=eval_df,
                             image_folder='data/train/train',
                             transform=train_transform)


# Setup to data loader
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
eval_loader = DataLoader(eval_ds, batch_size=256, shuffle=False)

In [ ]:
X, y = next(iter(train_loader))
X.shape, y.shape

## Define Model

In [ ]:
class ImageClassifier(torch.nn.Module):

    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.fc = torch.nn.Linear(1000, 1)


    def forward(self, x):
        x = self.backbone(x)
        x = self.fc(x)
        return x

In [ ]:
# Calling Backbone Model
backbone = convnext_large()

# Load Model
pth_file = 'models/convnext_large-ea097f82.pth'  # Replace with your .pth file path
state_dict = torch.load(pth_file, map_location=torch.device('cpu'))

# Load to Backbone
backbone.load_state_dict(state_dict)

In [ ]:
model = ImageClassifier(backbone)
for param in model.backbone.parameters():
    param.requires_grad = False

In [ ]:
model = model.to(device)

### Hyperparameter Tuning

In [ ]:
# Check if model can return as a class
dummy_input = torch.rand(1,3,224,224)
out = model(dummy_input)
out.shape

In [ ]:
# Hyperparameters
BATCH_SIZE = 256
device = "cuda"
num_epochs = 25
lr = 3e-4
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs*2, eta_min=5e-5)
threshold = 0.7
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True ,num_workers=4 ,pin_memory=True)
eval_loader = DataLoader(eval_ds, batch_size=BATCH_SIZE, shuffle=False ,num_workers=4 ,pin_memory=True)

### Training Loop

In [ ]:
# Training loop
train_loss_log = []
train_f1_log = []
val_loss_log = []
val_f1_log = []


for epoch in range(num_epochs):


  # Set Train Mode
  model.train()
  train_loss = 0.0
  train_total = 0
  train_preds = []
  train_labels = []


  # --------------- Training Loop ---------------
  for inputs, labels in tqdm(train_loader,
                             desc=f"Epoch {epoch+1}/{num_epochs}",
                             colour="green"):

    # Set Input and Labels to Devices
    inputs = inputs.to(device)
    labels = labels.to(device)

    # zero_grad -> Reset the gradients of all optimized
    optimizer.zero_grad()

    # Autocast -> Instances of autocast serve as context managers or decorators that allow regions of your script to run in mixed precision.
    with torch.autocast(device_type=device, dtype=torch.float16):
        outputs = model(inputs)
        loss = criterion(outputs, labels)

    # Loss Backward
    loss.backward()
    optimizer.step()

    # Calculation Loss
    train_loss += loss.item() * inputs.size(0)

    # Output Probability
    probs = torch.sigmoid(outputs)
    preds = (probs > threshold).float()
    train_total += labels.size(0)

    # Append to list
    train_preds.append(preds.cpu().numpy())
    train_labels.append(labels.cpu().numpy())


  train_loss /= len(train_loader.dataset)
  train_preds = np.vstack(train_preds)
  train_labels = np.vstack(train_labels)
  train_f1 = accuracy_score(train_labels, train_preds)
  train_loss_log.append(train_loss)
  train_f1_log.append(train_f1)


  # --------------- Validation ---------------
  model.eval()
  val_loss = 0.0
  val_total = 0
  val_preds = []
  val_labels = []

  with torch.no_grad():

      for inputs, labels in tqdm(eval_loader, desc="Validating", colour="yellow"):

          inputs = inputs.to(device)
          labels = labels.to(device)

          with torch.autocast(device_type=device, dtype=torch.float16):
              outputs = model(inputs)
              loss = criterion(outputs, labels)

          val_loss += loss.item() * inputs.size(0)
          probs = torch.sigmoid(outputs)
          preds = (probs > threshold).float()

          val_total += labels.size(0)
          val_preds.append(preds.cpu().numpy())
          val_labels.append(labels.cpu().numpy())


  val_loss /= len(eval_loader.dataset)

  # vstack -> Stack arrays in sequence vertically (row wise).
  val_preds = np.vstack(val_preds)
  val_labels = np.vstack(val_labels)
  val_f1 = accuracy_score(val_labels, val_preds)

  # Append to validation list
  val_loss_log.append(val_loss)
  val_f1_log.append(val_f1)

  # Update learning rate
  scheduler.step()

  # Print epoch results
  if epoch % 5 == 0:
      print(f"😼{"-" * 50}😼")
      print(f"Epoch {epoch+1}/{num_epochs}:")
      print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_f1:.4f}")
      print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_f1:.4f}")
      print(f"Learning Rate: {scheduler.get_last_lr()[0]:.6f}")
      print(" ")


## Setup Submission Project

In [ ]:
class TestImageDataset(Dataset):

    def __init__(self, dataframe,image_folder, transform=None):
        self.data = dataframe
        self.root = Path(image_folder)
        self.transform = transform
        self.image_paths = self.data['image_name'].values

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # Load image
        img_path = Path(self.image_paths[idx]+".jpg")
        image = Image.open(self.root/img_path).convert('RGB')  # Convert to RGB

        # Apply transformations
        if self.transform:
            image = self.transform(image)

        return image , self.image_paths[idx]

In [ ]:
test_df = pd.DataFrame({'image_name': glob('data/test/test/*.jpg')})
test_df["image_name"] = test_df.apply(lambda x: Path(x['image_name']).stem, axis=1)
test_df

In [ ]:
# Set Test Image Dataset
test_ds = TestImageDataset(dataframe=test_df,
                           image_folder='data/test/test',
                           transform=train_transform)

# Set to DataLoader -> DataLoader wraps an iterable around the Dataset to enable easy access to the samples.
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

In [ ]:
submission = {'id': [],"answer": []}

# Evaluation Mode -> switch for some specific layers/parts of the model that behave differently during training and inference (evaluating) time. For example, Dropouts Layers, BatchNorm Layers etc. You need to turn them off during model evaluation
model.eval()

# common practice for evaluating/validation is using torch.no_grad() in pair with model.eval() to turn off gradients computation
with torch.no_grad():

    for inputs, images_name in tqdm(test_loader, desc="Validating", colour="yellow"):
        inputs = inputs.to(device)
        outputs = model(inputs)
        probs = torch.sigmoid(outputs).squeeze()
        preds = (probs > threshold).float()
        submission["id"].extend(images_name)
        submission["answer"].extend(preds.cpu().numpy())

In [ ]:
submission_file = pd.DataFrame(submission)
submission_file

In [ ]:
submission_file.to_csv("submission/submission.csv",index=False)